<a href="https://colab.research.google.com/github/Ariqueeezz/Enterprise-Brand-Sentiment-and-Campaign-Analytics-Pipeline/blob/epic%2FEBSCAP-1/EBSCAP_1_EBSCAP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **EBSCAP-3: Download dan standardisasi dataset skala besar**

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
output_dir = '/content/drive/MyDrive/EBSCAP/data/raw'
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


In [13]:
import requests
from tqdm import tqdm

categories = ["Electronics", "Office_Products"]
base_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories"

def download_file(url, dest_path):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))

    with open(dest_path, 'wb') as f, tqdm(
        desc=os.path.basename(dest_path),
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            size = f.write(chunk)
            bar.update(size)

for cat in categories:
    url = f"{base_url}/{cat}.jsonl.gz"
    dest = os.path.join(output_dir, f"{cat}.jsonl.gz")
    if os.path.exists(dest):
        print(f"Skip {cat}, sudah ada.")
        continue
    print(f"Downloading {cat}...")
    download_file(url, dest)

Electronics.jsonl.gz: 100%|██████████| 6.03G/6.03G [01:56<00:00, 55.4MiB/s]


Office_Products.jsonl.gz: 100%|██████████| 1.51G/1.51G [00:35<00:00, 45.9MiB/s]


In [14]:
total_bytes = 0
for cat in categories:
    path = os.path.join(output_dir, f"{cat}.jsonl.gz")
    size = os.path.getsize(path)
    total_bytes += size
    print(f"{cat}: {size / (1024**3):.2f} GB")

print(f"\nTotal: {total_bytes / (1024**3):.2f} GB")

Electronics: 6.03 GB
Office_Products: 1.51 GB

Total: 7.54 GB


In [17]:
import json
import gzip

sample_file = os.path.join(output_dir, f"{categories[0]}.jsonl.gz")
with gzip.open(sample_file, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(json.loads(line))
        if i >= 2:
            break

{'rating': 3.0, 'title': 'Smells like gasoline! Going back!', 'text': 'First & most offensive: they reek of gasoline so if you are sensitive/allergic to petroleum products like I am you will want to pass on these.  Second: the phone adapter is useless as-is. Mine was not drilled far enough to be able to tighten it into place for my iPhone 12 max. It just slipped & slid all over. Stupid me putting the adapter together first without picking up the binoculars to smell them bc I wasted 15 minutes trying to figure out how to put the adapter together bc it does not come with instructions!  I had to come back here to the website which was a total pain. Third: the tripod is also useless. I would not trust the iOS to hold my $1600 phone nor even a Mattel Barbie for that matter. It’s just inefficient for the job imo.  Third: in order to try to give an honest review I did don gloves & eyewear to check the binoculars out.  They seemed average except for mine seemed to be missing about 10% of the f